# inference
#### This notebook runs inference with an ensemble of the Vision Transformer (ViT) and FaceNet models.

## Installing Dependencies

In [23]:
!pip uninstall -f pillow
!pip install pillow --upgrade
!pip install facenet-pytorch==2.5.3 --upgrade


Usage:   
  pip3 uninstall [options] <package> ...
  pip3 uninstall [options] -r <requirements file> ...

no such option: -f


## Imports

In [24]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from scipy.stats import mode
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from facenet_pytorch import InceptionResnetV1
import timm  # For ViT model
from facenet_pytorch import InceptionResnetV1

## Configurations

In [26]:
class Config:
    BASE_DIR = "/kaggle/input/surveillance-for-retail-stores/face_identification/face_identification"
    TRAIN_CSV = os.path.join(BASE_DIR, "trainset.csv")
    TEST_CSV = os.path.join(BASE_DIR, "eval_set.csv")
    FACENET_CHECKPOINT_DIR = "/kaggle/input/facenetepoch30/best_model_epoch_30.pth"
    VIT_CHECKPOINT_DIR = "/kaggle/input/vit-aggressive-finetuned/finetune_checkpoints/fold_3_aggressive_finetuned.pth" 
    OUTPUT_CSV = '/kaggle/working/predictions_late_fusion.csv'
    BATCH_SIZE = 512
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EMBEDDING_DIM = 1024
    IMAGE_SIZE = (160, 160)  # FaceNet uses 160x160; ViT uses 224x224, so we'll adapt
    NUM_WORKERS = 2
    SCALING_FACTOR = 500.0
    NUM_CLASSES = 125

## Dataset Helper Class

In [27]:
class FaceDataset(Dataset):
    def __init__(self, df, transform=None, is_train=True):
        self.df = df
        self.transform = transform
        self.is_train = is_train
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        try:
            img_path = self.df.iloc[idx]['full_path']
            image = Image.open(img_path).convert('RGB')
            image = np.array(image)
            if self.transform:
                augmented = self.transform(image=image)
                image = augmented['image']
            if self.is_train:
                label = self.df.iloc[idx]['label']
            else:
                label = -1
            return image, label
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            return self.__getitem__((idx + 1) % len(self.df))

## Defining Models -> ViT + FaceNet

In [28]:
# FaceNet Model
class FaceNetModel(nn.Module):
    def __init__(self, embedding_dim=1024):
        super(FaceNetModel, self).__init__()
        self.backbone = InceptionResnetV1(pretrained='vggface2')
        self.backbone.classify = False
        self.embedding_layer = nn.Sequential(
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, embedding_dim),
            nn.BatchNorm1d(embedding_dim)
        )
    
    def forward(self, x, return_embeddings=True):
        features = self.backbone(x)
        embeddings = self.embedding_layer(features)
        scaled_embeddings = embeddings * Config.SCALING_FACTOR
        return scaled_embeddings

# ViT Model
class AdvancedFaceReIDModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0)
        self.embedding_layer = nn.Sequential(
            nn.Linear(768, 2048),
            nn.BatchNorm1d(2048),
            nn.PReLU(),
            nn.Dropout(0.5),
            nn.Linear(2048, Config.EMBEDDING_DIM)
        )
    
    def forward(self, x):
        features = self.backbone(x)
        embeddings = self.embedding_layer(features)
        return embeddings

## Late Fusion Ensembled inference 

In [39]:
def inference(facenet_model, vit_model, train_df, test_df, le, facenet_weight=0.7, threshold=0.28):
    facenet_model.eval()
    vit_model.eval()
    
    # Define transforms
    facenet_transform = A.Compose([
        A.Resize(160, 160),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2()
    ])
    vit_transform = A.Compose([
        A.Resize(224, 224),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])
    
    # Load test data
    print("Loading test data...")
    test_df['image_path'] = test_df['image_path'].str.strip()
    test_df['full_path'] = test_df['image_path'].apply(lambda x: os.path.join(Config.BASE_DIR, 'test', x))
    print(f"Test data loaded: {len(test_df)} samples")
    
    test_dataset_facenet = FaceDataset(test_df, transform=facenet_transform, is_train=False)
    test_dataset_vit = FaceDataset(test_df, transform=vit_transform, is_train=False)
    test_loader_facenet = DataLoader(test_dataset_facenet, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS, pin_memory=True)
    test_loader_vit = DataLoader(test_dataset_vit, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS, pin_memory=True)
    
    # Generate reference embeddings separately for each model
    print("Generating reference embeddings...")
    ref_embeddings_facenet = {}
    ref_embeddings_vit = {}
    train_dataset_facenet = FaceDataset(train_df, transform=facenet_transform, is_train=True)
    train_dataset_vit = FaceDataset(train_df, transform=vit_transform, is_train=True)
    train_loader_facenet = DataLoader(train_dataset_facenet, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS, pin_memory=True)
    train_loader_vit = DataLoader(train_dataset_vit, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS, pin_memory=True)
    
    with torch.no_grad():
        for (images_facenet, labels), (images_vit, _) in zip(tqdm(train_loader_facenet, desc="FaceNet Ref"), train_loader_vit):
            images_facenet = images_facenet.to(Config.DEVICE)
            images_vit = images_vit.to(Config.DEVICE)
            embeddings_facenet = facenet_model(images_facenet)
            embeddings_vit = vit_model(images_vit)
            embeddings_facenet = nn.functional.normalize(embeddings_facenet, p=2, dim=1)
            embeddings_vit = nn.functional.normalize(embeddings_vit, p=2, dim=1)
            
            for emb_f, emb_v, lbl in zip(embeddings_facenet.cpu().numpy(), embeddings_vit.cpu().numpy(), labels.numpy()):
                gt = le.inverse_transform([lbl])[0]
                if gt not in ref_embeddings_facenet:
                    ref_embeddings_facenet[gt] = []
                    ref_embeddings_vit[gt] = []
                ref_embeddings_facenet[gt].append(emb_f)
                ref_embeddings_vit[gt].append(emb_v)
    
    for gt in ref_embeddings_facenet:
        ref_embeddings_facenet[gt] = np.mean(ref_embeddings_facenet[gt], axis=0)
        ref_embeddings_vit[gt] = np.mean(ref_embeddings_vit[gt], axis=0)
    print(f"Reference embeddings generated for {len(ref_embeddings_facenet)} identities")
    
    # Generate test embeddings and predict with late fusion
    print("Generating test embeddings and predicting with late fusion...")
    predictions = []
    total_predictions = 0
    doesnt_exist_count = 0
    
    with torch.no_grad():
        for (images_facenet, _), (images_vit, _) in zip(tqdm(test_loader_facenet, desc="FaceNet Test"), test_loader_vit):
            images_facenet = images_facenet.to(Config.DEVICE)
            images_vit = images_vit.to(Config.DEVICE)
            embeddings_facenet = facenet_model(images_facenet)
            embeddings_vit = vit_model(images_vit)
            embeddings_facenet = nn.functional.normalize(embeddings_facenet, p=2, dim=1)
            embeddings_vit = nn.functional.normalize(embeddings_vit, p=2, dim=1)
            embeddings_facenet = embeddings_facenet.cpu().numpy()
            embeddings_vit = embeddings_vit.cpu().numpy()
            
            batch_paths = test_df['full_path'][total_predictions:total_predictions+len(embeddings_facenet)]
            for emb_f, emb_v, full_path in zip(embeddings_facenet, embeddings_vit, batch_paths):
                # Compute distances separately
                distances_facenet = {gt: np.linalg.norm(emb_f - ref_emb) for gt, ref_emb in ref_embeddings_facenet.items()}
                distances_vit = {gt: np.linalg.norm(emb_v - ref_emb) for gt, ref_emb in ref_embeddings_vit.items()}
                
                # Combine distances with late fusion
                combined_distances = {}
                for gt in distances_facenet:
                    combined_dist = facenet_weight * distances_facenet[gt] + (1 - facenet_weight) * distances_vit[gt]
                    combined_distances[gt] = combined_dist
                
                min_dist = min(combined_distances.values())
                closest_gt = min(combined_distances, key=combined_distances.get)
                
                if min_dist <= threshold:
                    predicted_gt = closest_gt
                else:
                    predicted_gt = "doesn't_exist"
                    doesnt_exist_count += 1

                predictions.append({"image": os.path.basename(full_path), "gt": predicted_gt, "threshold": min_dist})
                total_predictions += 1

                
    # Print results
    proportion = doesnt_exist_count / total_predictions if total_predictions > 0 else 0
    print(f"\nResults:")
    print(f"Total Predictions: {total_predictions}")
    print(f"Doesn't Exist Count: {doesnt_exist_count}")
    print(f"Doesn't Exist Proportion: {proportion:.4f}")
    
    # Create submission file
    predictions_df = pd.DataFrame(predictions, columns=['image', 'gt', 'threshold'])
    predictions_df.to_csv(Config.OUTPUT_CSV, index=False)
    
    return predictions

## Main

In [40]:
def run_inference(threshold=0.28, facenet_weight=0.7):
    # Load FaceNet model
    facenet_checkpoint = Config.FACENET_CHECKPOINT_DIR
    facenet_model = FaceNetModel(embedding_dim=Config.EMBEDDING_DIM).to(Config.DEVICE)
    facenet_checkpoint_data = torch.load(facenet_checkpoint, map_location=Config.DEVICE)
    facenet_model.load_state_dict(facenet_checkpoint_data['model_state_dict'])
    print(f"Loaded FaceNet model from {facenet_checkpoint}")
    
    # Load ViT model
    vit_checkpoint = Config.VIT_CHECKPOINT_DIR
    vit_model = AdvancedFaceReIDModel(num_classes=Config.NUM_CLASSES).to(Config.DEVICE)
    vit_checkpoint_data = torch.load(vit_checkpoint, map_location=Config.DEVICE)
    state_dict = vit_checkpoint_data['model_state']  
    model_dict = vit_model.state_dict()
    filtered_state_dict = {k: v for k, v in state_dict.items() if k in model_dict}
    vit_model.load_state_dict(filtered_state_dict, strict=False)
    print(f"Loaded ViT model from {vit_checkpoint}")
    
    # Load training and test data
    print("Loading training data...")
    train_df = pd.read_csv(Config.TRAIN_CSV, sep=',', engine='python', header=0, names=['image_path', 'gt'], on_bad_lines='skip')
    train_df = train_df.dropna()
    train_df['image_path'] = train_df['image_path'].str.strip()
    train_df['gt'] = train_df['gt'].str.strip()
    train_df['full_path'] = train_df['image_path'].apply(lambda x: os.path.join(Config.BASE_DIR, x))
    le = LabelEncoder()
    train_df['label'] = le.fit_transform(train_df['gt'])
    print(f"Training data loaded: {len(train_df)} samples, {len(le.classes_)} classes")
    
    test_df = pd.read_csv(Config.TEST_CSV)
    
    predictions = inference(facenet_model, vit_model, train_df, test_df, le, facenet_weight=facenet_weight, threshold=threshold)
    return predictions

In [41]:
if __name__ == "__main__":
    threshold = 0.435  
    facenet_weight = 0.69 
    predictions = run_inference(threshold=threshold, facenet_weight=facenet_weight)

/usr/local/lib/python3.10/dist-packages/facenet_pytorch/models/inception_resnet_v1.py:329: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(cached_file)

Loaded FaceNet model from /kaggle/input/facenetepoch30/best_model_epoch_30.pth


<ipython-input-40-1e5663dbf8b5>:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  vit_checkpoint_data = torch.load(vit_checkpoint, map_location=Config.DEVICE)


Loaded ViT model from /kaggle/input/vit-aggressive-finetuned/finetune_checkpoints/fold_3_aggressive_finetuned.pth
Loading training data...
Training data loaded: 6828 samples, 125 classes
Loading test data...
Test data loaded: 4734 samples
Generating reference embeddings...


FaceNet Ref: 100%|██████████| 14/14 [01:40<00:00,  7.18s/it]


Reference embeddings generated for 125 identities
Generating test embeddings and predicting with late fusion...


FaceNet Test: 100%|██████████| 10/10 [01:18<00:00,  7.85s/it]


Results:
Total Predictions: 4734
Doesn't Exist Count: 2009
Doesn't Exist Proportion: 0.4244
